# MLB exploration

Pull team stats, fetch live sportsbook odds, and compute a betting edge.

Run cells in order. The odds cell needs `ODDS_API_KEY` in `.env` — the
other cells work offline (first run of pybaseball hits the network).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
pd.set_option('display.max_columns', 30)

## 1. Team batting for last season

In [ ]:
from src.mlb.data import get_team_batting

bat = get_team_batting(2024)
print(bat.shape)
bat.head()

## 2. Today's MLB lines (needs ODDS_API_KEY)

In [ ]:
from src.mlb.odds import get_mlb_odds, OddsAPIError

try:
    lines = get_mlb_odds()
    print(lines.shape)
    display(lines.head(10))
except OddsAPIError as e:
    print('Skipping odds (no API key):', e)

## 3. Compute edge vs a market price

Plug in your model's win probability and the American odds you see at
a book. Positive edge = +EV (given your model).

In [ ]:
from src.mlb.analysis import implied_probability, edge, kelly_fraction

model_prob = 0.58   # e.g. your estimate for the home team
market     = -130   # American odds at the book

print('market implied prob :', round(implied_probability(market), 4))
print('edge (per $1 stake) :', round(edge(model_prob, market), 4))
print('full Kelly fraction :', round(kelly_fraction(model_prob, market), 4))